In [1]:

# %%
import h5py
from annbatch import DatasetCollection
import anndata as ad
from annbatch.samplers import RandomSampler
from annbatch import Loader
import shutil
import numpy as np
import pandas as pd
import os

import cellink

from annbatch import write_sharded

import zarr



In [2]:
def load_only_X_lazy(group):
    return ad.AnnData(
        X=ad.experimental.read_lazy(group["X"]),
        obs=ad.io.read_elem(group["obs"]),
        var=ad.io.read_elem(group["var"]),
    )


In [ ]:

def load_obs_from_random_sampler(sampler, obs):
    "gives the unique donor ids per batch."
    for lr in sampler.sample(len(obs)):
        indices = np.concatenate([np.arange(x.start,x.stop) for x in lr["requests"]])
        sampled_obs = obs.iloc[indices]
        for split in lr["splits"]:
            yield sampled_obs.iloc[split].unique()


# %%
from annbatch.abc import Sampler
from annbatch.utils import split_given_size
import math
import numpy as np

class DonorSampler(Sampler):
    """Custom sampler to yield donor observations in a specific sequence,
    loading preload_nbatches batches from disk at a time to minimize I/O overhead.
    """
    
    def __init__(self, categories, obs_names, preload_nbatches=4):
        self._batch_size = 1
        self._preload_nbatches = preload_nbatches
        self._categories = list(categories)
        
        # Map category names to their corresponding integer indices in dd_G
        name_to_idx = {name: i for i, name in enumerate(obs_names)}
        self.indices = [name_to_idx[cat] for cat in self._categories if cat in name_to_idx]
        
    @property
    def batch_size(self) -> int:
        return self._batch_size
        
    @property
    def shuffle(self) -> bool:
        return False
        
    def n_batches(self, n_obs: int) -> int:
        return math.ceil(len(self.indices) / self._batch_size)
        
    def validate(self, n_obs: int) -> None:
        if any(idx >= n_obs or idx < 0 for idx in self.indices):
            raise ValueError("Some mapped indices are out of bounds for the dataset.")
            
    def _sample(self, n_obs: int):
        step_size = self._preload_nbatches * self._batch_size
        for i in range(0, len(self.indices), step_size):
            block_indices = self.indices[i : i + step_size]
            
            # Load the block of rows from disk
            chunks = [slice(idx, idx + 1) for idx in block_indices]
            
            # Split the preloaded rows into batches of size batch_size
            splits = split_given_size(np.arange(len(block_indices)), self._batch_size)
            
            yield {
                "requests": chunks,
                "splits": splits,
            }



# %%

def train_yields(n_epochs):

    rs1 = RandomSampler(
        batch_size=4, chunk_size=8, preload_nchunks=4,
        replacement=False, drop_last=True,
        rng=np.random.default_rng(42)    
    )
    rs2 = RandomSampler(
        batch_size=4, chunk_size=8, preload_nchunks=4,
        replacement=False, drop_last=True,
        rng=np.random.default_rng(42)    
    )

    loader_C = Loader(
        return_index=True,
        batch_sampler=rs1,
    ).use_collection(DatasetCollection("dd_C_collection.zarr"))

    obs = pd.concat(x["donor_id"] for x in loader_C._obs)
    
    # Get observation names (donor IDs) of dd_G from the collection
    dd_G_collection = DatasetCollection("dd_G_collection.zarr")
    dd_G_obs_names = []
    for group in dd_G_collection:
        dd_G_obs_names.extend(ad.io.read_elem(group["obs"]).index.tolist())
    
    for epoch in range(n_epochs):
        flattened_donors = []
        n_donors_per_batch = []
        for batch_donors in load_obs_from_random_sampler(rs1, obs):
            flattened_donors.extend(batch_donors)
            n_donors_per_batch.append(len(batch_donors))
        
        donor_sampler = DonorSampler(categories=flattened_donors, obs_names=dd_G_obs_names, preload_nbatches=4)
        loader_G = Loader(
            return_index=True,
            batch_sampler=donor_sampler,
        ).use_collection(dd_G_collection)
        loader_G_iter = iter(loader_G)

        n_donors_per_batch_iter = iter(n_donors_per_batch)
        for batch in loader_C:
            n_donors = next(n_donors_per_batch_iter)
            donors_g = []
            for _ in range(n_donors):
                batch_g = next(loader_G_iter)
                donors_g.append(batch_g)
            batch["donor_g"] = donors_g
            yield batch


In [17]:
a = train_yields(1)

In [18]:
b = next(a)

In [22]:
b

{'X': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 3922 stored elements and shape (4, 36469)>,
 'obs':                     orig.ident  nCount_RNA  nFeature_RNA  percent.mt  \
 barcode                                                                
 ACGCCGAAGCCAGGAT-55     onek1k      6218.0          1790    2.155034   
 TACTTGTAGCCAGAAC-45     onek1k      1642.0           712    2.679659   
 AAACCTGTCCATGCTC-65     onek1k      2190.0           704    1.506849   
 CAAGATCCAGGAATGC-55     onek1k      1646.0           717    3.280680   
 
                        donor_id pool_number predicted.celltype.l2  \
 barcode                                                             
 ACGCCGAAGCCAGGAT-55   OneK1K_53          55             CD16 Mono   
 TACTTGTAGCCAGAAC-45  OneK1K_469          45                   gdT   
 AAACCTGTCCATGCTC-65  OneK1K_191          65                  Treg   
 CAAGATCCAGGAATGC-55   OneK1K_47          55                   gdT   
 
                  